# Can teacher training copy a behaviour the model lacked, faithfully and with Δ fixed?

*E42, `experiments/teacher_training/newskill.py`, 2026-10-28. Record:
`reports/analyses/2026-10-28_analysis_e42-teacher-new-behaviour.md`. Plan:
`reports/plans/2026-10-28_plan_e42-teacher-new-behaviour.md`. Follows E10, E12 and E34.*

## The answer

Yes by the paper's checks, with one caveat that matters. We copied fine-tunes that teach the model a word cipher. The
copies kept the new behaviour on all three tasks, passed the adversarial check on ordinary text and left Δ untouched.
But with Γ frozen, the model built only from the subcomponents Γ marks important at the trigger loses almost all of it.
This holds on both random ciphers and under every map we read. Unexpectedly, a copy trained on plain Pile text alone learnt the
one-pair cipher, which we filed it would not.

## What we were trying to test

Teacher training copies a fine-tune into the decomposition. The subcomponents' read and write vectors are trained so
that the model built from them imitates the fine-tuned model, with Γ and Δ frozen. In E34 it kept every fine-tune's
behaviour and passed the adversarial check. But every fine-tune it copied taught something the model already half did.
`GOALS.md` asks what happens when post-training teaches something new. It expects that with Δ fixed, a new behaviour
will run through subcomponents Γ keeps off.

The new behaviour is E10's word cipher. Each window of text holds a marker token that never occurs in the source text,
then a common word, the key. The fine-tuned model must put the key's value next. Each task has 403 training windows
and 97 held-out ones:

- **32 random pairs**: each of 32 keys maps to an arbitrary other word, a substitution cipher the model lacks.
- **1 random pair**: one key and its arbitrary value.
- **Identity control**: each of the 32 keys is followed by itself. The model half does this already, and in E10 its gain
  came from Γ switching on subcomponents that already copy. It checks whether what we see on the random pairs belongs to
  the format or to the behaviour being new.

The fine-tune is E10's, run longer: all 24 matrices, Adam, 1,000 steps. We copied it twice, 1,000 steps each. The copy
on task text trains on the task's training windows. The copy on Pile text trains on 8,192 rows of ordinary Pile text,
which hold no marker. It asks whether the fine-tune's predictions on ordinary text carry the new behaviour.

Each state is read three ways.

- **Behaviour**: the held-out probability of the target at the trigger. The unchanged model reads about 0.00002 on the
  32 random pairs and 0.0002 on the other two.
- **Faithfulness**: the paper's adversarial check on 256 Pile rows. A 20-step adversary searches the masks for the
  largest output damage. We report what it finds at the state over what it finds at the unchanged model, so 1 means no
  change. On this L40 card the unchanged model varies by 0.21 across three adversary seeds. So a state passes at or
  below 1.24 and fails at or above 1.81.
- **Where the gain sits at the trigger**: the masked-off share. The gain is the rise in the target's log-odds at the
  trigger. The masked-off share is the part of it present when every subcomponent is switched on, but lost when Γ's
  causal-importance values mask them. At 0 the CI-masked model keeps the whole gain; at 1 it keeps none.

Two references sit beside every copy. **Do-nothing** leaves the whole fine-tune in Δ. The **least-squares split**
writes the fine-tune as the least-norm change of the subcomponents and puts the remainder in Δ. It should fail the
check, which shows the check can fail on these tasks.

## What we expected, and why

- **The copy on task text keeps the behaviour and passes, on every task.** Component training on the cipher's labels
  passed on the 32-pair cipher once it had settled. Every task-text copy in E34 passed.
- **Its gain runs through subcomponents Γ keeps off, on both random ciphers.** Label training put 0.75 and 0.86 of the
  gain there in E10. On the identity control we expected a small share, since Γ switches on subcomponents that already
  copy.
- **The copy on Pile text learns nothing of the cipher**, staying below 0.5 on every task. Pile text holds no marker,
  and the fine-tune's training also fitted its predictions at the windows' other positions. We held this with medium
  confidence: E34's Pile-text copies once learnt a behaviour we filed they would not.

## Does the copy keep the behaviour and pass the check, with Δ untouched?

![copy keeps the behaviour and passes](../experiments/teacher_training/figures/e42_copy.png)

**Reading it.** Yes, on all three tasks. In the left panel every copy keeps the target's probability at 0.96 or more,
far above the unchanged model's dashed line. In the right panel the copies read 0.96 to 0.99, inside the unchanged
model's own seed band. The least-squares split fails on every task, so the check can fail here. Do-nothing reads
between the lines on all three. Δ took none of the copy's change, as it must when frozen. On held-out windows the
copy's predictions sit within 1% of the fine-tune's, measured against the unchanged model's distance from it. One
caveat: the split fails only just on two tasks, at 1.83 and 1.86.

## Does the CI-masked model carry the new behaviour?

![CI-masked model on Pile text and at the trigger](../experiments/teacher_training/figures/e42_trigger.png)

**Reading it.** On ordinary text yes; at the trigger no, on both random ciphers. The left panel is the paper's CI-masked
check on Pile text, where every state reads within 4% of the unchanged model. The right panel reads the same masked
model at the trigger, on held-out windows. The copy on 32 random pairs puts 0.96 on the target with Δ included. With
every subcomponent on and Δ left out it puts 0.95 there. Masked by Γ's causal-importance values, it puts 0.007. On one
pair the masked model keeps 0.001 of a full 1.00. On the identity control it keeps 0.93 of 0.98. Do-nothing and the
least-squares split read the same at the trigger. So with Γ frozen this belongs to the task, as E10 found, not to
teacher training. Pile text cannot show it: the marker occurred nowhere in the 1,024 Pile rows we scanned.

## Does the copy's gain run through subcomponents Γ keeps off?

![masked-off share at the trigger](../experiments/teacher_training/figures/e42_routes.png)

**Reading it.** Yes on both random ciphers. The masked-off share of the copy's gain at the trigger is 0.84 on the large
cipher and 0.92 on the one-pair cipher. Label-trained component training read 0.75 and 0.86 in E10, so the copy goes the same way and a
little further. On the identity control the share is 0.31, just above the 0.25 line we filed as "not masked-off". There
most of the gain comes from Γ switching on subcomponents that already copy, as with label training. Do-nothing reads
below zero by construction: its whole update sits in Δ, outside the masked model. Caveat: one batch order, and how far
these shares move between batch orders is unmeasured.

## Can ordinary Pile text alone teach the copy the cipher?

![copy trained on Pile text, and the fine-tune's predictions on Pile text](../experiments/teacher_training/figures/e42_pile.png)

**Reading it.** For one pair, yes, which we filed it would not. The copy trained on Pile text reached 0.91 on the
one-pair cipher by step 1,000, and passed the check at 1.10. On the other two tasks it stayed far below 0.5, as filed:
0.02 and 0.001. The right panel shows a likely reason. In plain Pile text, with no marker, the
fine-tune puts 0.31 on the value right after the one-pair key, where the unchanged model puts 0.0001. That key occurs
182 times in the first 1,024 rows. For 32 pairs the fine-tune's leak is far smaller, at 0.005. Whether this leak is
what teaches the copy is untested. Caveat: 0.91 sits only 0.01 above the line. Pile-text copies moved by up to 0.03
between batch orders in E34, so a rerun could land "between", which would still contradict the filed
prediction.

In [1]:
# The scored numbers behind the figures, from the three results files (no model).
import json
from pathlib import Path

res = Path("../experiments/teacher_training/results")
print(f"{'task':8s} {'copy P':>7s} {'check':>6s} {'masked-off':>10s} {'Pile-only P':>11s}")
for t in ("rand32", "rand1", "ident32"):
    r = json.loads((res / f"teacher_training_e42_{t}.json").read_text())
    a, b = r["A"]["1000"], r["B"]["1000"]
    print(f"{t:8s} {a['p_target']:7.3f} {a['pgd']:6.2f} {a['share_e_gate']:10.2f} {b['p_target']:11.3f}")

task      copy P  check masked-off Pile-only P
rand32     0.964   0.96       0.84       0.019
rand1      1.000   0.99       0.92       0.911
ident32    0.983   0.96       0.31       0.001


## What we learned

Teacher training copies a behaviour the model lacked. On three cipher tasks it keeps the behaviour, passes the paper's
checks on ordinary text and leaves Δ alone. With Γ frozen, the copy carries the new behaviour through subcomponents Γ
keeps off at the trigger. That is what `GOALS.md`'s "create" branch expects when Δ may not move. So the weak version's
verdict hangs on which of two sentences in `GOALS.md` decides it. The success criterion of 2026-10-19 says the paper's
checks decide and the masked-off share describes mechanism; by it, E42 passes. The other sentence says the weak version
holds when the masked-off and residual parts are small. By that one it fails on both random ciphers. The run cannot
choose between them. Separately, the text-only copy can learn a new behaviour when the fine-tune shows it in ordinary text.

## How compelling is the evidence

- **One draw of everything.** One batch order, one fine-tune seed, and per state one adversary start of 20 steps on 256
  Pile rows. In E34 a second random start once read a fine-tune six times higher, so passes are lower bounds.
- **The L40's band is wide**, from 1.24 to 1.81; an H100's is about a third as wide. No copy depends on it: every copy reads
  1.10 or less.
- **The check's power is thin on two tasks.** The least-squares split fails at 1.83 and 1.86, just over the line.
- **The trigger readings** are means over 97 held-out windows at one position, logged to three decimals. How far the
  masked-off share moves between batch orders is unmeasured.
- **The fine-tune reproduced E10's** to four decimals on 32 pairs and the identity control. On one pair that check could
  not fail: E10 read 0.998, and any working fine-tune lands within 0.02 of it.
- **Scope**: the 67M model, three tasks of one format, one fine-tune recipe.

## Surprises

- **Something unexpected happened that I cannot fully explain.** The copy trained on Pile text alone learnt the one-pair
  cipher. We filed it would stay below 0.5; it reached 0.91 and passed. A likely cause is visible but untested: the
  fine-tune shows the pair in ordinary text.
- **The fine-tune leaks its map into plain text more than the plan expected.** The plan expected at most 10 times the
  unchanged model's probability. It read 146 times on the large cipher, and about 3,000 times on the one-pair cipher.
  Reported, not scored.
- **The identity control's masked-off share landed at 0.31**, just above the 0.25 line, so "mixed" where we leaned "not
  masked-off". A narrow miss: at step 100 it read 0.25.

## What next

1. **List the subcomponents that carry the copy's gain** on the random ciphers, from the saved states on CPU, at no
   cost. Suppose a handful carry it and Γ is near zero on them at the trigger. Then a small change to Γ is the natural
   repair; if hundreds share it, it is not.
2. **Test the Pile-only one-pair result** by dropping every Pile row that holds the key. If the copy still learns, the
   fine-tune's leak at the key is not the cause.
3. **The user's decision** on whether the CI-masked check on the task's own inputs joins the criterion. If it does, the
   next step on new behaviours is updating Γ, not another map with Γ frozen.

## Scorecard

Filed in `experiments/teacher_training/results/predictions_e42.json` before any run, unedited. Scored by the run, under
`score` in each results file, and checked here against the filed rules.

| id | claim | rule | read | verdict |
|---|---|---|---|---|
| C1 | unchanged model under the Pile harness | W₀ at its known values, floor ≤ stop | floor 0.209, pass | pass |
| G1 | fine-tune at step 10 reproduces E10 | within 0.02 of E10 | 0.5847 / 0.9595 / 0.9982 | pass ×3 (one pair cannot fail) |
| P1 rand32 | task-text copy faithful | `state_verdict` at 1000 | P 0.964, ratios 0.960 / 0.936 | **pass**, as predicted |
| P1 rand1 | task-text copy faithful | same | P 1.000, ratios 0.989 / 0.941 | **pass**, as predicted |
| P1 ident32 | task-text copy faithful | same | P 0.983, ratios 0.957 / 0.929 | **pass**, as predicted |
| P2 rand32 | masked-off | e_gate ≥ 0.5 | 0.839 | **masked-off**, lean held |
| P2 rand1 | masked-off | e_gate ≥ 0.5 | 0.918 | **masked-off**, lean held |
| P2 ident32 | not masked-off | e_gate ≤ 0.25 | 0.309 | **mixed**: lean missed narrowly |
| P3 rand32 | Pile-text copy: neither | P < 0.5 | P 0.019 | **neither**, lean held |
| P3 rand1 | Pile-text copy: neither | P < 0.5 | P 0.911, ratios 1.095 / 0.883, pass | **learnt: reversed** |
| P3 ident32 | Pile-text copy: neither | P < 0.5 | P 0.0013 | **neither**, lean held |
| power | least-squares split fails | verdict fail | 1.977 / 1.826 / 1.855 | power on all three |

## Not run

- A second batch order or fine-tune seed; a second adversary start or a longer adversary on any state.
- An H100 rerun: the decision tree sends inconclusive copies there, and no copy was inconclusive.
- The list of subcomponents that carry the copy's gain, a branch of the plan, from the saved states.
- The CI-masked KL at the trigger itself, rather than the target's probability; the unchanged model's own trigger cells
  were not kept in the results.
- The Pile-only one-pair copy without the key's rows; Pile-text copies longer than 1,000 steps.
- A Pile reading of the copies; an SGD fine-tune or copy; a LoRA fine-tune; E10's step-10 fine-tune; the larger model.

## Provenance

| task | cluster job | run, 2026-10-28 | results file |
|---|---|---|---|
| 32 random pairs | 40166 | 12:02-12:23 | `results/teacher_training_e42_rand32.json` |
| identity control | 40194 | 12:12-12:35 | `results/teacher_training_e42_ident32.json` |
| 1 random pair | 40195 | 12:14-12:36 | `results/teacher_training_e42_rand1.json` |

MATS cluster, NVIDIA L40, debug QoS, no cost. Logs in `cluster/logs/commute-teacher-e42-<job>.out`, saved states in
`experiments/teacher_training/results/large/e42/<task>/`, figures from `experiments/teacher_training/plot_e42.py`.